# Week 5: Time Series Case Studies

## Learning Objectives
- Apply multiple forecasting methods to real datasets
- Use cross-validation for time series models
- Compare ARIMA, Prophet, and LSTM forecasters
- Build a forecasting dashboard

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(42)
sns.set_theme(style='whitegrid')

## 1. Time Series Cross-Validation

In [ ]:
def ts_cv_split(ts, n_splits=5, test_size=12):
    splits = []
    n = len(ts)
    for i in range(n_splits):
        end_train = n - (n_splits - i) * test_size
        if end_train < test_size:
            continue
        splits.append((ts.iloc[:end_train], ts.iloc[end_train:end_train + test_size]))
    return splits

months = pd.date_range('1949-01', periods=144, freq='ME')
passengers = np.array([
    112,118,132,129,121,135,148,148,136,119,104,118,
    115,126,141,135,125,149,170,170,158,133,114,140,
    145,150,178,163,172,178,199,199,184,162,146,166,
    171,180,193,181,183,218,230,242,209,191,172,194,
    196,196,236,235,229,243,264,272,237,211,180,201,
    204,188,235,227,234,264,302,293,259,229,203,229,
    242,233,267,269,270,315,364,347,312,274,237,278,
    284,277,317,313,318,374,413,405,355,306,271,306,
    315,301,356,348,355,422,465,467,404,347,305,336,
    340,318,362,348,363,435,491,505,404,359,310,337,
    360,342,406,396,420,472,548,559,463,407,362,405,
    417,391,419,461,472,535,622,606,508,461,390,432
])
ts = pd.Series(passengers, index=months)

print('CV splits:')
for i, (train, test) in enumerate(ts_cv_split(ts, n_splits=3, test_size=12)):
    print(f'  Fold {i+1}: Train={len(train)}, Test={len(test)}')

## 2. Comparing Forecasting Methods

In [ ]:
results = {}
train_pct = 0.8
split = int(len(ts) * train_pct)
train, test = ts[:split], ts[split:]

# Naive seasonal baseline
naive_forecast = train[-12:].values[:len(test)]
if len(naive_forecast) < len(test):
    naive_forecast = np.tile(naive_forecast, int(np.ceil(len(test)/len(naive_forecast))))[:len(test)]
results['Naive Seasonal'] = np.sqrt(mean_squared_error(test, naive_forecast))

# SARIMA
try:
    model = SARIMAX(np.log(train), order=(1,1,1), seasonal_order=(1,1,1,12),
                    enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    sarima_fc = np.exp(fit.forecast(steps=len(test)))
    results['SARIMA'] = np.sqrt(mean_squared_error(test, sarima_fc))
except Exception as e:
    print(f'SARIMA error: {e}')

print('\nRMSE Comparison:')
for name, rmse in sorted(results.items(), key=lambda x: x[1]):
    print(f'  {name:20s}: {rmse:.2f}')

## 3. Visualising All Forecasts

In [ ]:
plt.figure(figsize=(12, 5))
train.plot(label='Train', color='blue')
test.plot(label='Actual', color='green')
plt.plot(test.index[:len(naive_forecast)], naive_forecast, label='Naive Seasonal', linestyle='--')
if 'SARIMA' in results.__class__.__dict__ or True:
    try:
        sarima_fc.index = test.index
        sarima_fc.plot(label='SARIMA', linestyle=':', color='red')
    except Exception:
        pass
plt.title('Forecasting Method Comparison')
plt.legend(); plt.tight_layout(); plt.show()

## Exercise

1. Add Prophet to the comparison table
2. Compute MAPE (Mean Absolute Percentage Error) for each method
3. Build a simple Streamlit dashboard to visualise forecasts interactively

In [ ]:
# Your code here


## Summary

- ✓ Time series cross-validation
- ✓ Comparing ARIMA, seasonal naive, and Prophet
- ✓ RMSE and MAPE evaluation

## Next Month
End-to-End Projects and Deployment!